### First set of imports 

In [1]:
import pandas as pd 
import json
import os
import numpy as np
import torch
import re

pd.set_option("display.max_colwidth", None) # Just allows me to see full columns for dataframes




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/ubuntu/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/ubuntu/.local/lib/python3.10/site-pac

### Convert videos json to df

In [2]:
#Get the videos json
videos = os.path.join("..", "videos_all_languages.json")


#Open the json
with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

#Turn the json into a pandas dataframe with proper headings     
df = pd.json_normalize(data["videos"])

print(df.columns)
print(df.head())

### Get all the handlandmark paths 

Index(['id', 'language', 'Sign', 'filepath', 'HamNoSys', 'concept_url',
       'HandLandmark', 'annotated_video'],
      dtype='object')
      id language      Sign  \
0  BSL_1      BSL     April   
1  BSL_2      BSL    Athens   
2  BSL_3      BSL    August   
3  BSL_4      BSL    Berlin   
4  BSL_5      BSL  February   

                                            filepath  \
0     /ephemeral/Project/Data/External/BSL/April.mp4   
1    /ephemeral/Project/Data/External/BSL/Athens.mp4   
2    /ephemeral/Project/Data/External/BSL/August.mp4   
3    /ephemeral/Project/Data/External/BSL/Berlin.mp4   
4  /ephemeral/Project/Data/External/BSL/February.mp4   

                                          HamNoSys  \
0     
1                 
2         
3                                         
4                             

              

In [3]:
#find the path to each landmark json in the videos json
Hand_Landmarks_paths = df["HandLandmark"]

Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]



In [4]:

REDUCED_FACE_LANDMARKS = (
    list(range(70, 103)) +   # eyebrows
    list(range(0, 17)) +     # outer lips
    list(range(61, 88)) +    # inner lips
    list(range(152, 172))    # jawline
)

HAND_LANDMARK_IDS = list(range(21))  # 21 per hand
POSE_LANDMARK_IDS = list(range(33))  # 33 pose points

def convert_frames_to_sequence(frames):
    rows = []

    for frame in frames:
        frame_features = []

        # ---------------- HANDS ----------------
        # Build dicts for left and right hand
        left = {lm["id"]: lm for hand in frame["hands"] if hand["handedness"] == "Left" for lm in hand["landmarks"]}
        right = {lm["id"]: lm for hand in frame["hands"] if hand["handedness"] == "Right" for lm in hand["landmarks"]}

        # Left hand (21 landmarks)
        for i in HAND_LANDMARK_IDS:
            if i in left:
                lm = left[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        # Right hand (21 landmarks)
        for i in HAND_LANDMARK_IDS:
            if i in right:
                lm = right[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        # ---------------- POSE ----------------
        pose_dict = {lm["id"]: lm for pose in frame["pose"] for lm in pose["landmarks"]}

        for i in POSE_LANDMARK_IDS:
            if i in pose_dict:
                lm = pose_dict[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        # ---------------- FACE (reduced) ----------------
        face_dict = {lm["id"]: lm for face in frame["face"] for lm in face["landmarks"]}

        for i in REDUCED_FACE_LANDMARKS:
            if i in face_dict:
                lm = face_dict[i]
                frame_features.extend([lm["x"], lm["y"], lm["z"]])
            else:
                frame_features.extend([0.0, 0.0, 0.0])

        rows.append(frame_features)

    seq = np.array(rows, dtype=np.float32)

    # Downsample
    #seq = seq[::3]

    return seq


In [5]:
import unicodedata

def fix_mojibake(name):
    # 1. Try the classic UTF-8 → Latin-1 fix
    try:
        name = name.encode("latin1").decode("utf8")
    except:
        pass

    # 2. Manual replacements for common mojibake patterns
    replacements = {
        # German
        "ÃŸ": "ß", "Ã„": "Ä", "Ã–": "Ö", "Ãœ": "Ü",
        "Ã¤": "ä", "Ã¶": "ö", "Ã¼": "ü",

        # French
        "Ã©": "é", "Ã¨": "è", "Ãª": "ê", "Ã«": "ë",
        "Ã´": "ô", "Ã¢": "â", "Ã¹": "ù", "Ã»": "û",
        "Ã§": "ç",

        # Greek (common mojibake)
        "Î±": "α", "Î²": "β", "Î³": "γ", "Î´": "δ",
        "Îµ": "ε", "Î¶": "ζ", "Î·": "η", "Î¸": "θ",
        "Î¹": "ι", "Îº": "κ", "Î»": "λ", "Î¼": "μ",
        "Î½": "ν", "Î¾": "ξ", "Î¿": "ο", "Îπ": "π",
        "Ïƒ": "σ", "Ï„": "τ", "Ï…": "υ", "Ï†": "φ",
        "Ï‡": "χ", "Ïˆ": "ψ", "Ï‰": "ω",
        "Î ": "Π", "Î£": "Σ", "Î¤": "Τ", "Î¨": "Ψ",
        "Î©": "Ω",
    }

    for bad, good in replacements.items():
        name = name.replace(bad, good)

    # 3. Normalise Unicode (fixes combining accents)
    name = unicodedata.normalize("NFC", name)

    return name


In [6]:
os.makedirs("processed_sequences", exist_ok=True)

for path in Hand_Landmarks:
    with open(path, "r") as f:
        data = json.load(f)

    raw = data["video_path"]

    video_id = fix_mojibake(
        os.path.basename(raw)
        .replace(".mp4", "")
        .replace(".MOV", "")
        .replace("_cropped", "")
    )

    seq = convert_frames_to_sequence(data["frames"])

    np.save(f"processed_sequences/{video_id}.npy", seq)


In [7]:
df["filepath"] = (
    df["filepath"]
    .apply(lambda p: fix_mojibake(os.path.basename(p).replace(".mp4", "").replace(".MOV", "")))
)



In [8]:
label_dict = dict(zip(df["filepath"], df["HamNoSys"]))


X = []
y = []

for seq_file in os.listdir("processed_sequences"):
    vid = seq_file.replace(".npy", "")
    clean_vid = fix_mojibake(vid)

    if clean_vid in label_dict:
        X.append(np.load(f"processed_sequences/{seq_file}"))
        y.append(label_dict[clean_vid])
    else:
        print("Missing label for:", vid)


In [9]:
max_len = max(seq.shape[0] for seq in X)
num_features = X[0].shape[1]

x = np.zeros((len(X), max_len, num_features), dtype=np.float32)

for i, seq in enumerate(X):
    x[i, :seq.shape[0], :] = seq


In [10]:
X_tensor = torch.tensor(x, dtype=torch.float32)


In [11]:
all_tokens = sorted({c for seq in y for c in seq})
token_to_id = {t: i+3 for i, t in enumerate(all_tokens)}
token_to_id["<PAD>"] = 0
token_to_id["<SOS>"] = 1
token_to_id["<EOS>"] = 2

vocab_size = len(token_to_id)
id_to_token = {v: k for k, v in token_to_id.items()}



In [12]:
y_sequences = []

for seq in y:
    encoded = [token_to_id["<SOS>"]] + [token_to_id[c] for c in seq] + [token_to_id["<EOS>"]]
    y_sequences.append(encoded)


In [13]:
import torch
import torchvision
import torchaudio
import lightning as L

print(torch.__version__)
print(torchvision.__version__)
print(torchaudio.__version__)


2.2.2+cu121
0.17.2+cu121
2.2.2+cu121


In [14]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam #Do I keep adam or GD
import matplotlib.pyplot as plt
import seaborn as sns
import lightning as L
from torch.utils.data import TensorDataset, DataLoader
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np
import Levenshtein as Lev


import random
from sklearn.model_selection import train_test_split
import math 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

sos_id = token_to_id["<SOS>"]
eos_id = token_to_id["<EOS>"]
pad_id = token_to_id["<PAD>"]

print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")




Using device: cuda
True
1
NVIDIA H100 PCIe


In [15]:
npy_ids = [seq_file.replace(".npy", "") for seq_file in os.listdir("processed_sequences")]

missing = [vid for vid in npy_ids if fix_mojibake(vid) not in label_dict]

print("Missing labels:", len(missing))
print(missing[:30])


Missing labels: 0
[]


### Splitting Data into Train, Val, Test

In [16]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_sequences, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

### Trims rubbish from sequence

### Creates windows so that we can expand dataset

In [17]:
def temporal_subsample(seq, rates=[2, 3], min_frames=20):
    augmented = []
    for r in rates:
        new_seq = seq[::r]
        if new_seq.shape[0] >= min_frames:
            augmented.append(new_seq)
    return augmented


def random_frame_drop(seq, drop_prob=0.05, min_frames=20):
    mask = np.random.rand(seq.shape[0]) > drop_prob
    new_seq = seq[mask]
    return new_seq if new_seq.shape[0] >= min_frames else None



# -----------------------------
# APPLY TEMPORAL SUBSAMPLING + FRAME DROP
# -----------------------------
X_aug = []
y_aug = []

MIN_FRAMES = 20

for x_seq, y_seq in zip(X_train, y_train):

    # original
    if x_seq.shape[0] >= MIN_FRAMES:
        X_aug.append(x_seq)
        y_aug.append(y_seq)

    # subsampled
    for new_seq in temporal_subsample(x_seq, min_frames=MIN_FRAMES):
        X_aug.append(new_seq)
        y_aug.append(y_seq)

    # frame drop
    dropped = random_frame_drop(x_seq, min_frames=MIN_FRAMES)
    if dropped is not None:
        X_aug.append(dropped)
        y_aug.append(y_seq)



In [ ]:

def pad_sequences_X(seqs, max_len):
    num_features = seqs[0].shape[1]
    X = torch.zeros((len(seqs), max_len, num_features), dtype=torch.float32)

    for i, seq in enumerate(seqs):
        length = min(seq.shape[0], max_len)
        X[i, :length] = torch.tensor(seq[:length], dtype=torch.float32)

    return X


def pad_sequences_y(labels, max_len, pad_token=0):
    Y = torch.full((len(labels), max_len), pad_token, dtype=torch.long)

    for i, seq in enumerate(labels):
        length = min(len(seq), max_len)
        Y[i, :length] = torch.tensor(seq[:length], dtype=torch.long)

    return Y


max_len_X = max(seq.shape[0] for seq in X_aug + X_val + X_test)
max_len_y = max(len(seq) for seq in y_aug + y_val + y_test)


X_train_tensor = pad_sequences_X(X_aug, max_len_X)
y_train_tensor = pad_sequences_y(y_aug, max_len_y)

X_val_tensor = pad_sequences_X(X_val, max_len_X)
y_val_tensor = pad_sequences_y(y_val, max_len_y)

X_test_tensor = pad_sequences_X(X_test, max_len_X)
y_test_tensor = pad_sequences_y(y_test, max_len_y)




### Wraps Tensors

In [19]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

### Batches of 16 of the wrapped Tensors

In [20]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset,batch_size=16)

### Encoder Class - Summary of frames for decoder class

In [21]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


In [ ]:
class EncoderBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            bidirectional=True,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, x):

        lengths = (x.abs().sum(dim=-1) != 0).sum(dim=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_outputs, _ = self.lstm(packed)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs, batch_first=True, total_length=x.size(1)
        )


        outputs = self.fc(outputs)

        return outputs


In [23]:
class CustomTransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.2):
        super().__init__()

        self.self_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )

        self.linear1 = nn.Linear(d_model, d_model * 4)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_model * 4, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):

        # Self-attention
        _tgt, _ = self.self_attn(
            tgt, tgt, tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask
        )
        tgt = self.norm1(tgt + self.dropout1(_tgt))

        # Cross-attention
        _tgt, cross_attn_weights = self.cross_attn(
            tgt, memory, memory,
            attn_mask=None,
            key_padding_mask=memory_key_padding_mask,
            need_weights=True,
            average_attn_weights=False
        )


        tgt = self.norm2(tgt + self.dropout2(_tgt))

        # Feed-forward
        ff = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        tgt = self.norm3(tgt + self.dropout3(ff))

        return tgt, cross_attn_weights



### Decoder Class - Looks at the memory and assigns tokens

In [24]:
class CustomTransformerDecoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=4, num_heads=8, dropout=0.2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.positional_encoding = PositionalEncoding(hidden_dim)

        self.layers = nn.ModuleList([
            CustomTransformerDecoderLayer(hidden_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tgt, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask):

        x = self.embedding(tgt)
        pe = self.positional_encoding.pe[:, :x.size(1)].to(x.device)
        mask = (~tgt_key_padding_mask).unsqueeze(-1).to(x.device)
        x = x + pe * mask


        cross_attn_all_layers = []

        for layer in self.layers:
            x, cross_attn = layer(
                x, memory,
                tgt_mask,
                tgt_key_padding_mask,
                memory_key_padding_mask
            )

            cross_attn_all_layers.append(cross_attn)

        logits = self.fc_out(x)
        return logits, cross_attn_all_layers

### Loss function

In [25]:
criterion = nn.CrossEntropyLoss(ignore_index=token_to_id["<PAD>"], label_smoothing = 0.1) #Loss function

### Seq2seq - Runs encoder and decoder

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, input_dim, vocab_size, hidden_dim=256):
        super().__init__()
        self.encoder = EncoderBiLSTM(input_dim, hidden_dim)
        self.decoder = CustomTransformerDecoder(vocab_size, hidden_dim)

    def generate_square_subsequent_mask(self, size):
        mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
        return mask

    def forward(self, src, tgt):

        memory = self.encoder(src)


        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)

        tgt_key_padding_mask = (tgt == pad_id)

        src_key_padding_mask = (src.abs().sum(dim=-1) == 0)

        logits, cross_attn = self.decoder(
            tgt,
            memory,
            tgt_mask,
            tgt_key_padding_mask,
            src_key_padding_mask
        )

        return logits, cross_attn



In [ ]:
def compute_coverage_loss(cross_attn_layers):

    attn = cross_attn_layers[-1]  

    attn = attn.mean(dim=1)  

    batch, tgt_len, src_len = attn.shape
    
    coverage = torch.zeros(batch, src_len, device=attn.device)

    cov_loss = 0.0

    for t in range(tgt_len):
        a_t = attn[:, t, :]  

        cov_loss += torch.sum(torch.min(a_t, coverage))

        coverage = coverage + a_t

    cov_loss = cov_loss / batch

    return cov_loss



In [ ]:
num_features = X_tensor.shape[2]  
hidden_size = 256

model = Seq2Seq(
    input_dim=num_features,
    vocab_size=vocab_size,
    hidden_dim=256
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr = 5e-5)

def get_lr(epoch, base_lr=5e-5, warmup_epochs=8, max_epochs=120):
    if epoch < warmup_epochs:
        return base_lr * (epoch / warmup_epochs)
    else:
        progress = (epoch - warmup_epochs) / (max_epochs - warmup_epochs)
        return base_lr * 0.5 * (1 + math.cos(math.pi * progress))




### Updated weights based on loss function

In [29]:
def token_accuracy(output, trg, pad_idx=0):
    
    pred = output.argmax(1)
    mask = trg != pad_idx
    correct = (pred == trg) & mask
    accuracy = correct.sum().item()/mask.sum().item()
    return accuracy*100

In [38]:
def clean(seq, sos_id, eos_id, pad_id):
    seq = [t for t in seq if t not in (sos_id, pad_id)]
    if eos_id in seq:
        seq = seq[:seq.index(eos_id)]
    return seq



In [40]:
def beam_search(
    model,
    src,
    sos_id,
    eos_id,
    pad_id=0,
    beam_size=5,
    max_len=200,
    alpha=0.7
):
    device = src.device
    model.eval()

    memory = model.encoder(src)
    src_key_padding_mask = (src.abs().sum(dim=-1) == 0)

    beams = [([sos_id], 0.0)]

    for _ in range(max_len):
        candidates = []

        for seq, score in beams:
            if seq[-1] == eos_id:
                candidates.append((seq, score))
                continue

            tgt = torch.tensor(seq, device=device).unsqueeze(0)
            tgt_mask = model.generate_square_subsequent_mask(len(seq)).to(device)
            tgt_key_padding_mask = (tgt == pad_id)

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                tgt_key_padding_mask,
                src_key_padding_mask
            )

            log_probs = torch.log_softmax(logits[:, -1, :], dim=-1)
            topk = torch.topk(log_probs, beam_size)

            for k in range(beam_size):
                token = topk.indices[0, k].item()
                token_score = topk.values[0, k].item()

                new_seq = seq + [token]
                new_score = score + token_score

                lp = ((5 + len(new_seq)) / 6) ** alpha
                new_score = new_score / lp

                candidates.append((new_seq, new_score))

        beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_size]

        if all(seq[-1] == eos_id for seq, _ in beams):
            break

    return beams[0][0]


In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, epoch):
    model.train()
    total_loss = 0

    for src, tgt in dataloader:
        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()

        decoder_input = tgt[:, :-1]   

        logits, cross_attn = model(src, decoder_input)

        pred = logits.reshape(-1, logits.size(-1)) 
        gold = tgt[:, 1:].contiguous().view(-1)     
        assert pred.size(0) == gold.size(0), f"Shape mismatch: {pred.size(0)} vs {gold.size(0)}"

        non_pad_mask = (gold != pad_id)
        pred = pred[non_pad_mask]
        gold = gold[non_pad_mask]
        ce_loss = criterion(pred, gold)


        cov_loss = compute_coverage_loss(cross_attn)

        loss = ce_loss + 0.05 * cov_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:

num_epochs = 300
patience = 150
best_val_loss = float('inf')
best_cer = float('inf')
patience_counter = 0

os.makedirs("checkpoints", exist_ok=True)

print("Starting training...")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

 
    lr = get_lr(epoch)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

   
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        epoch
    )
    print(f"  Train Loss: {train_loss:.4f}")

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)

            logits, cross_attn = model(src, tgt[:, :-1])

            gold = tgt[:, 1:]
            mask = (gold != pad_id)[:, :logits.size(1)]

            pred = logits.reshape(-1, vocab_size)[mask.reshape(-1)]
            gold = gold[mask]

            loss = criterion(pred, gold)

            val_loss += loss.item()

    val_loss /= len(val_loader)
    print(f"  Val Loss:   {val_loss:.4f}")


    cer_list = []

    with torch.no_grad():
        for src, tgt in val_loader:
            src = src.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):
                pred = beam_search(model, src[i].unsqueeze(0), sos_id, eos_id, pad_id)
                pred = clean(pred, sos_id, eos_id, pad_id)

                true = tgt[i].tolist()
                true = clean(true, sos_id, eos_id, pad_id)


                pred_str = "".join(id_to_token[t] for t in pred)
                true_str = "".join(id_to_token[t] for t in true)

                cer = Lev.distance(pred_str, true_str) / max(1, len(true_str))
                cer_list.append(cer)

    avg_cer = sum(cer_list) / len(cer_list)
    print(f"  CER:        {avg_cer:.4f}")


    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "checkpoints/best_token_model.pt")
        print("  ✔ Saved best token-level model (val loss improved).")
    else:
        patience_counter += 1
        print(f"  No val-loss improvement ({patience_counter}/{patience})")


    if avg_cer < best_cer:
        best_cer = avg_cer
        torch.save(model.state_dict(), "checkpoints/best_sequence_model.pt")
        print("  ✔ Saved best sequence-level model (CER improved).")


    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), f"checkpoints/model_epoch_{epoch+1}.pt")
        print(f"  ✔ Saved checkpoint at epoch {epoch+1}")

    if patience_counter >= patience:
        print("\nEarly stopping triggered.")
        break


torch.save(model.state_dict(), "checkpoints/last_model.pt")
print("\nTraining complete. Final model saved.")


Starting training...

Epoch 1/300
  Train Loss: 12.0288
  Val Loss:   5.3875
  CER:        17.2703
  ✔ Saved best token-level model (val loss improved).
  ✔ Saved best sequence-level model (CER improved).

Epoch 2/300
  Train Loss: 11.1176
  Val Loss:   4.0412
  CER:        17.1631
  ✔ Saved best token-level model (val loss improved).
  ✔ Saved best sequence-level model (CER improved).

Epoch 3/300
  Train Loss: 10.3488
  Val Loss:   3.3953
  CER:        3.8087
  ✔ Saved best token-level model (val loss improved).
  ✔ Saved best sequence-level model (CER improved).

Epoch 4/300
  Train Loss: 9.8335
  Val Loss:   3.1127
  CER:        2.4797
  ✔ Saved best token-level model (val loss improved).
  ✔ Saved best sequence-level model (CER improved).

Epoch 5/300
  Train Loss: 9.5456
  Val Loss:   2.9708
  CER:        7.2048
  ✔ Saved best token-level model (val loss improved).

Epoch 6/300
  Train Loss: 9.1476
  Val Loss:   2.9227
  CER:        6.4894
  ✔ Saved best token-level model (val lo

In [ ]:
def decode_ids(id_list, id_to_token):
    return [id_to_token[i] for i in id_list]



In [ ]:
def evaluate_model(model, loader):
    model.eval()
    
    preds = []
    trues = []

    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):
                pred_seq = beam_search(model, src[i].unsqueeze(0), sos_id, eos_id, pad_id)
                pred_seq = clean(pred_seq, sos_id, eos_id, pad_id)

                true_seq = tgt[i].tolist()
                true_seq = clean(true_seq, sos_id, eos_id, pad_id)

                preds.append(pred_seq)
                trues.append(true_seq)

    decoded_preds = [decode_ids(p, id_to_token) for p in preds]
    decoded_trues = [decode_ids(t, id_to_token) for t in trues]

    bleu_scores = []
    cer_scores = []
    exact = 0

    for pred, true in zip(decoded_preds, decoded_trues):
        bleu_scores.append(compute_bleu(pred, true))
        cer_scores.append(compute_cer(pred, true))
        if pred == true:
            exact += 1

    avg_bleu = {
        "BLEU-1": np.mean([b["BLEU-1"] for b in bleu_scores]),
        "BLEU-2": np.mean([b["BLEU-2"] for b in bleu_scores]),
        "BLEU-4": np.mean([b["BLEU-4"] for b in bleu_scores]),
    }

    avg_cer = np.mean(cer_scores)
    exact_acc = exact / len(decoded_preds)

    return avg_bleu, avg_cer, exact_acc

### Validation

In [ ]:
import sys
print(sys.executable)


In [ ]:
smooth = SmoothingFunction().method1

def compute_bleu(pred, true):
    return {
        "BLEU-1": sentence_bleu([true], pred, weights=(1,0,0,0), smoothing_function=smooth),
        "BLEU-2": sentence_bleu([true], pred, weights=(0.5,0.5,0,0), smoothing_function=smooth),
        "BLEU-4": sentence_bleu([true], pred, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth),
    }


In [ ]:
def compute_cer(pred, true):
    pred_str = "".join(pred)
    true_str = "".join(true)
    if len(true_str) == 0:
        return 1.0 if len(pred_str) > 0 else 0.0
    return Lev.distance(pred_str, true_str) / len(true_str)


In [ ]:
model.load_state_dict(torch.load("checkpoints/best_sequence_model.pt"))
model.eval()

avg_bleu, avg_cer, exact_acc = evaluate_model(model, val_loader)
print(f" avg_bleu, {avg_cer:.2%}, exact_acc")


In [ ]:
import matplotlib.pyplot as plt

def evaluate_checkpoint(path):
    model.load_state_dict(torch.load(path))
    model.eval()
    bleu, cer, exact = evaluate_model(model, val_loader)
    return bleu, cer, exact

checkpoints = sorted([f"checkpoints/{f}" for f in os.listdir("checkpoints") if f.endswith(".pt")])

cer_values = []
bleu1_values = []
bleu4_values = []
names = []

for ckpt in checkpoints:
    bleu, cer, exact = evaluate_checkpoint(ckpt)
    cer_values.append(cer)
    bleu1_values.append(bleu["BLEU-1"])
    bleu4_values.append(bleu["BLEU-4"])
    names.append(ckpt)

plt.figure(figsize=(14,5))
plt.plot(cer_values, label="CER")
plt.plot(bleu1_values, label="BLEU-1")
plt.plot(bleu4_values, label="BLEU-4")
plt.xticks(range(len(names)), names, rotation=90)
plt.legend()
plt.title("Checkpoint Evaluation")
plt.show()


In [ ]:
def show_examples(model, loader, n=10):
    model.eval()
    examples = []

    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):
                pred = beam_search(model, src[i].unsqueeze(0), sos_id, eos_id, pad_id)
                if eos_id in pred:
                    pred = pred[:pred.index(eos_id)]

                true = tgt[i].tolist()
                if eos_id in true:
                    true = true[:true.index(eos_id)]

                pred_str = "".join(id_to_token[t] for t in pred)
                true_str = "".join(id_to_token[t] for t in true)

                examples.append((true_str, pred_str))

                if len(examples) >= n:
                    return examples

model.load_state_dict(torch.load("checkpoints/best_sequence_model.pt"))

examples = show_examples(model, val_loader, n=10)

for i, (true, pred) in enumerate(examples):
    print(f"\nExample {i+1}")
    print("TRUE:", true)
    print("PRED:", pred)
